# 천안시 통합 격자통계 생성

통합 시설 CSV와 100m 인구 추정 CSV를 이용하여 500m 격자통계를 생성합니다.

Colab에서는 Google Drive의 `MyDrive/Cheonan`을 사용하고, 로컬에서는 현재 프로젝트 폴더를 사용합니다.

## 1. 입력·출력 설정

노트북을 프로젝트 폴더에서 실행한다는 전제로 상대경로를 사용합니다.


In [1]:
from pathlib import Path
import math
import re
import struct
import zipfile

import numpy as np
import pandas as pd
from pyproj import Transformer

# Colab에서는 Google Drive를 마운트하고, 로컬에서는 마운트하지 않습니다.
IS_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IS_COLAB = True
except ModuleNotFoundError:
    pass

# 로컬 노트북을 Cheonan 프로젝트 폴더가 아닌 곳에서 실행할 때만 직접 지정합니다.
LOCAL_PROJECT_DIR = None
if IS_COLAB:
    BASE_DIR = Path('/content/drive/MyDrive/Cheonan')
elif LOCAL_PROJECT_DIR is not None:
    BASE_DIR = Path(LOCAL_PROJECT_DIR)
else:
    BASE_DIR = Path.cwd()

INPUT_DIR = BASE_DIR / 'grid_inputs'
POPULATION_INPUT = BASE_DIR / 'grid_outputs' / 'cheonan_sgis_100m_final_features_with_population.csv'
FACILITY_INPUT = INPUT_DIR / 'cheonan_all_facilities_geocoded.csv'
BOUNDARY_ZIP = INPUT_DIR / 'cheonan_adm_dong_boundary_20250630.zip'
OUTPUT_DIR = BASE_DIR / 'grid_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRID_SIZES = [500]
PROJECTED_CRS = 'EPSG:5186'
WGS84_CRS = 'EPSG:4326'
TO_PROJECTED = Transformer.from_crs(WGS84_CRS, PROJECTED_CRS, always_xy=True)
TO_WGS84 = Transformer.from_crs(PROJECTED_CRS, WGS84_CRS, always_xy=True)

required_inputs = [POPULATION_INPUT, FACILITY_INPUT, BOUNDARY_ZIP]
missing_inputs = [path for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError('다음 입력 파일이 없습니다:\n' + '\n'.join(map(str, missing_inputs)))

print(f'실행 환경: {"Google Colab" if IS_COLAB else "로컬"}')
print(f'프로젝트 폴더: {BASE_DIR}')
print(f'시설 입력: {FACILITY_INPUT}')
print(f'인구 입력: {POPULATION_INPUT}')
print(f'경계 입력: {BOUNDARY_ZIP}')
print(f'출력 폴더: {OUTPUT_DIR}')


실행 환경: 로컬
프로젝트 폴더: C:\Users\심현석\Documents\test\Cheonan-0825
시설 입력: C:\Users\심현석\Documents\test\Cheonan-0825\grid_inputs\cheonan_all_facilities_geocoded.csv
인구 입력: C:\Users\심현석\Documents\test\Cheonan-0825\grid_outputs\cheonan_sgis_100m_final_features_with_population.csv
경계 입력: C:\Users\심현석\Documents\test\Cheonan-0825\grid_inputs\cheonan_adm_dong_boundary_20250630.zip
출력 폴더: C:\Users\심현석\Documents\test\Cheonan-0825\grid_outputs


## 2. 행정동 경계에서 격자 생성

원본 파이프라인의 경계 판정 논리를 유지합니다. 각 격자의 중심점이 천안시 행정동 경계 안에 있는 경우만 사용합니다.


In [2]:
def read_dbf_records(dbf_bytes: bytes, encoding: str = "cp949") -> list[dict[str, str]]:
    """DBF 바이너리에서 행정동 속성 테이블을 읽습니다."""
    nrec = struct.unpack("<I", dbf_bytes[4:8])[0]
    header_len = struct.unpack("<H", dbf_bytes[8:10])[0]
    rec_len = struct.unpack("<H", dbf_bytes[10:12])[0]
    fields = []
    offset = 32
    while dbf_bytes[offset] != 0x0D:
        desc = dbf_bytes[offset:offset + 32]
        name = desc[:11].split(b"\x00", 1)[0].decode("ascii", "ignore")
        fields.append((name, desc[16]))
        offset += 32
    rows = []
    for i in range(nrec):
        row = dbf_bytes[header_len + i * rec_len: header_len + (i + 1) * rec_len]
        pos = 1
        values = {}
        for name, size in fields:
            values[name] = row[pos:pos + size].decode(encoding, "ignore").strip()
            pos += size
        rows.append(values)
    return rows


def read_boundary_shapes(path: Path) -> list[dict]:
    """행정동 ZIP의 SHP/DBF를 읽어 다각형 링과 속성을 반환합니다."""
    with zipfile.ZipFile(path) as archive:
        shp_name = next(name for name in archive.namelist() if name.endswith(".shp"))
        dbf_name = next(name for name in archive.namelist() if name.endswith(".dbf"))
        shp = archive.read(shp_name)
        attrs = read_dbf_records(archive.read(dbf_name))

    shapes = []
    offset = 100
    while offset + 8 <= len(shp):
        record_no, content_words = struct.unpack(">2i", shp[offset:offset + 8])
        offset += 8
        content = memoryview(shp)[offset:offset + content_words * 2]
        offset += content_words * 2
        if len(content) < 44 or struct.unpack("<i", content[0:4])[0] != 5:
            continue
        num_parts, num_points = struct.unpack("<2i", content[36:44])
        parts_start = 44
        parts = list(struct.unpack("<" + "i" * num_parts, content[parts_start:parts_start + 4 * num_parts]))
        point_start = parts_start + 4 * num_parts
        points = [struct.unpack("<2d", content[point_start + 16 * i:point_start + 16 * (i + 1)]) for i in range(num_points)]
        rings = []
        for i, start in enumerate(parts):
            end = parts[i + 1] if i + 1 < len(parts) else num_points
            ring = points[start:end]
            if len(ring) >= 3:
                rings.append({"points": ring, "bbox": (min(x for x, _ in ring), min(y for _, y in ring), max(x for x, _ in ring), max(y for _, y in ring))})
        shapes.append({"attrs": attrs[record_no - 1], "bbox": struct.unpack("<4d", content[4:36]), "rings": rings})
    return shapes


def point_in_ring(x: float, y: float, ring: list[tuple[float, float]]) -> bool:
    inside = False
    j = len(ring) - 1
    for i, (xi, yi) in enumerate(ring):
        xj, yj = ring[j]
        if (yi > y) != (yj > y):
            x_intersection = (xj - xi) * (y - yi) / ((yj - yi) or 1e-30) + xi
            if x < x_intersection:
                inside = not inside
        j = i
    return inside


def point_in_shape(x: float, y: float, shape: dict) -> bool:
    min_x, min_y, max_x, max_y = shape["bbox"]
    if x < min_x or x > max_x or y < min_y or y > max_y:
        return False
    inside = False
    for ring in shape["rings"]:
        rx0, ry0, rx1, ry1 = ring["bbox"]
        if x < rx0 or x > rx1 or y < ry0 or y > ry1:
            continue
        if point_in_ring(x, y, ring["points"]):
            inside = not inside
    return inside


def find_admin(x: float, y: float, shapes: list[dict]) -> dict | None:
    for shape in shapes:
        if point_in_shape(x, y, shape):
            return shape["attrs"]
    return None


def create_grid(grid_m: int, shapes: list[dict]) -> pd.DataFrame:
    """지정한 크기의 격자를 만들고 중심 행정동을 부여합니다."""
    all_x = [x for shape in shapes for ring in shape["rings"] for x, _ in ring["points"]]
    all_y = [y for shape in shapes for ring in shape["rings"] for _, y in ring["points"]]
    x0 = math.floor(min(all_x) / grid_m) * grid_m
    x1 = math.ceil(max(all_x) / grid_m) * grid_m
    y0 = math.floor(min(all_y) / grid_m) * grid_m
    y1 = math.ceil(max(all_y) / grid_m) * grid_m

    rows = []
    for y in range(y0, y1, grid_m):
        for x in range(x0, x1, grid_m):
            center_x, center_y = x + grid_m / 2, y + grid_m / 2
            admin = find_admin(center_x, center_y, shapes)
            if admin is None:
                continue
            rows.append({
                "GRID_CD": f"G{grid_m}_{len(rows) + 1:05d}",
                "x_min": float(x), "y_min": float(y),
                "x_max": float(x + grid_m), "y_max": float(y + grid_m),
                "center_x": float(center_x), "center_y": float(center_y),
                "grid_m": grid_m, "selection": "center_in_cheonan",
                "center_adm_cd": admin.get("ADM_CD", ""),
                "center_adm_nm": admin.get("ADM_NM", ""),
            })
    grid = pd.DataFrame(rows)
    grid["center_lon"], grid["center_lat"] = TO_WGS84.transform(grid["center_x"].to_numpy(), grid["center_y"].to_numpy())
    return grid


## 3. 인구와 시설을 목표 격자에 집계

인구는 기존 100m 추정 격자의 중심점을 목표 격자에 포함시켜 합산합니다. 시설은 통합 CSV의 위·경도가 있는 행만 좌표 변환 후 집계합니다.


In [3]:
def safe_name(value: object) -> str:
    """중분류명을 CSV 컬럼명에 사용할 수 있도록 정리합니다."""
    text = re.sub(r"\s+", "_", str(value).strip())
    text = re.sub(r"[^0-9A-Za-z가-힣_]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_") or "unknown"


def add_population(grid: pd.DataFrame, population: pd.DataFrame, grid_m: int) -> pd.DataFrame:
    """100m 인구 추정 격자를 목표 격자별로 합산합니다."""
    pop = population.copy()
    pop["pop_value"] = pd.to_numeric(pop["pop_2026_est_int"], errors="coerce").fillna(0)
    pop["household_value"] = pd.to_numeric(pop["households_2026_est_int"], errors="coerce").fillna(0)
    pop["pop_2024_value"] = pd.to_numeric(pop["pop_2024_grid"], errors="coerce").fillna(0)
    px, py = TO_PROJECTED.transform(pop["center_lon"].to_numpy(), pop["center_lat"].to_numpy())
    pop["target_x_min"] = np.floor(px / grid_m) * grid_m
    pop["target_y_min"] = np.floor(py / grid_m) * grid_m
    pop_summary = pop.groupby(["target_x_min", "target_y_min"], as_index=False).agg(
        pop_2024_grid=("pop_2024_value", "sum"),
        pop_2026_est_int=("pop_value", "sum"),
        households_2026_est_int=("household_value", "sum"),
        population_source_grid_count=("GRID_CD", "count"),
    )
    result = grid.copy()
    result["target_x_min"] = result["x_min"]
    result["target_y_min"] = result["y_min"]
    result = result.merge(pop_summary, on=["target_x_min", "target_y_min"], how="left")
    for col in ["pop_2024_grid", "pop_2026_est_int", "households_2026_est_int", "population_source_grid_count"]:
        result[col] = result[col].fillna(0)
    result["pop_2026_est_int"] = result["pop_2026_est_int"].round().astype(int)
    result["households_2026_est_int"] = result["households_2026_est_int"].round().astype(int)
    result["population_source_grid_count"] = result["population_source_grid_count"].astype(int)
    return result.drop(columns=["target_x_min", "target_y_min"])


def add_facilities(grid: pd.DataFrame, facilities: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """통합 시설 CSV의 좌표를 목표 격자에 배정하고 대분류·중분류별 개수를 집계합니다."""
    result = grid.copy()
    facility = facilities.copy()
    facility["위도"] = pd.to_numeric(facility["위도"], errors="coerce")
    facility["경도"] = pd.to_numeric(facility["경도"], errors="coerce")
    valid = facility["위도"].between(30, 40) & facility["경도"].between(120, 135)
    facility = facility.loc[valid].copy()
    fx, fy = TO_PROJECTED.transform(facility["경도"].to_numpy(), facility["위도"].to_numpy())
    facility["x_min"] = np.floor(fx / int(grid["grid_m"].iloc[0])) * int(grid["grid_m"].iloc[0])
    facility["y_min"] = np.floor(fy / int(grid["grid_m"].iloc[0])) * int(grid["grid_m"].iloc[0])
    grouped = facility.groupby(["x_min", "y_min", "생활지수", "대분류", "중분류"], as_index=False).size().rename(columns={"size": "facility_count"})

    if not grouped.empty:
        for (index_name, major), sub in grouped.groupby(["생활지수", "대분류"]):
            total = sub.groupby(["x_min", "y_min"], as_index=False)["facility_count"].sum()
            total_col = f"facility_{safe_name(index_name)}__{safe_name(major)}__total"
            total = total.rename(columns={"facility_count": total_col})
            result = result.merge(total, left_on=["x_min", "y_min"], right_on=["x_min", "y_min"], how="left")
            for middle, middle_df in sub.groupby("중분류"):
                middle_sum = middle_df.groupby(["x_min", "y_min"], as_index=False)["facility_count"].sum()
                middle_col = f"facility_{safe_name(index_name)}__{safe_name(major)}__{safe_name(middle)}"
                middle_sum = middle_sum.rename(columns={"facility_count": middle_col})
                result = result.merge(middle_sum, on=["x_min", "y_min"], how="left")

    facility_cols = [col for col in result.columns if col.startswith("facility_")]
    result[facility_cols] = result[facility_cols].fillna(0)
    for col in facility_cols:
        result[col] = result[col].astype(int)
    audit = {
        "facility_rows_total": int(len(facilities)),
        "facility_rows_with_valid_coordinates": int(valid.sum()),
        "facility_rows_without_valid_coordinates": int((~valid).sum()),
    }
    return result, audit


def build_grid_statistics(grid_m: int, shapes: list[dict], population: pd.DataFrame, facilities: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    grid = create_grid(grid_m, shapes)
    grid = add_population(grid, population, grid_m)
    grid, facility_audit = add_facilities(grid, facilities)
    return grid, facility_audit

shapes = read_boundary_shapes(BOUNDARY_ZIP)
population = pd.read_csv(POPULATION_INPUT, encoding="utf-8-sig")
facilities = pd.read_csv(FACILITY_INPUT, encoding="utf-8-sig")

for grid_m in GRID_SIZES:
    grid_stats, audit = build_grid_statistics(grid_m, shapes, population, facilities)
    output_path = OUTPUT_DIR / f"cheonan_grid_{grid_m}m_integrated_features.csv"
    grid_stats.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"{grid_m}m 저장:", output_path)
    print("  격자 수:", len(grid_stats), "시설 좌표 유효 행:", audit["facility_rows_with_valid_coordinates"])


500m 저장: C:\Users\심현석\Documents\test\Cheonan-0825\grid_outputs\cheonan_grid_500m_integrated_features.csv
  격자 수: 2551 시설 좌표 유효 행: 2889


## 4. 결과 확인

두 격자통계의 행 수와 주요 인구·시설 컬럼을 확인합니다.


In [4]:
for grid_m in GRID_SIZES:
    path = OUTPUT_DIR / f"cheonan_grid_{grid_m}m_integrated_features.csv"
    check = pd.read_csv(path, encoding="utf-8-sig")
    facility_columns = [col for col in check.columns if col.startswith("facility_")]
    print(f"\n[{grid_m}m] shape={check.shape}, facility_columns={len(facility_columns)}")
    print("인구 합계:", int(check["pop_2026_est_int"].sum()))
    print("인구 있는 격자:", int((check["pop_2026_est_int"] > 0).sum()))
    print("시설 합계:", int(check[facility_columns].sum().sum()))



[500m] shape=(2551, 65), facility_columns=48
인구 합계: 664135
인구 있는 격자: 1758
시설 합계: 5774
